In [1]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import numpy as np
from matplotlib import pyplot as plt
import psutil

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

Device: cuda
RAM: 11.9%


In [2]:
# load features and edges, build PyG data object
feat = torch.load('../data/canwell_features.pt')
x        = feat['x']
y        = feat['y']
is_slope = feat['is_slope']
is_basin = feat['is_basin']
has_diff = feat['has_diff']

edge_index = torch.load('../data/canwell_edgidx.pt')

data = Data(x=x, edge_index=edge_index, y=y)
data.is_slope = is_slope
data.is_basin = is_basin
data.has_diff = has_diff

print(f"nodes: {data.num_nodes:,}")
print(f"edges: {data.num_edges:,}")
print(f"node features: {data.num_node_features}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

nodes: 4,792,230
edges: 38,302,670
node features: 4
ram: 19.2%


In [3]:
from sklearn.preprocessing import StandardScaler

# normalize node features
x_np = x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)
data.x = torch.tensor(x_scaled, dtype=torch.float)
print("data scaled")

data scaled


In [4]:
# define masks
# training nodes: has diff value AND is slope, randomly select 70%
slope_with_diff  = (is_slope & has_diff).nonzero(as_tuple=True)[0]
print(f"slope nodes with diff: {len(slope_with_diff):,}")

slope nodes with diff: 2,838,168


In [5]:
# random 70/15/15 on slopediff nodes
perm = torch.randperm(len(slope_with_diff))
n = len(slope_with_diff)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

train_idx = slope_with_diff[perm[:train_end]]
val_idx   = slope_with_diff[perm[train_end:val_end]]
test_idx  = slope_with_diff[perm[val_end:]]

# basin nodes never masked

print(f"train nodes: {len(train_idx):,}")
print(f"val nodes:   {len(val_idx):,}")
print(f"test nodes:  {len(test_idx):,}")
print(f"basin nodes: {is_basin.sum():,}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

train nodes: 1,986,717
val nodes:   425,725
test nodes:  425,726
basin nodes: 2,321,200
ram: 20.7%


In [6]:
# add masking feature and set up input

# add is_masked as 5th input feature
# train/val/test slope nodes mask=1 -- their diff is hidden from INPUT
# basin nodes get mask=0 -- diff visible as context, added as 6th feature

is_masked = torch.zeros(data.num_nodes, dtype=torch.float)
is_masked[train_idx] = 1.0
is_masked[val_idx] = 1.0
is_masked[test_idx] = 1.0

# diff as input feature: visible for basin nodes, 0 for masked (slope) nodes
diff_input = torch.zeros(data.num_nodes, dtype=torch.float)
diff_input[is_basin & has_diff] = y[is_basin & has_diff]

# normalize diff_input
diff_mean = diff_input[is_basin & has_diff].mean().item()
diff_std  = diff_input[is_basin & has_diff].std().item()
diff_input[is_basin & has_diff] = (diff_input[is_basin & has_diff] - diff_mean)/diff_std

# stack into final feature matrix:
#       [elev, slope, aspect, doubslope, is_masked, diff_input]

data.x = torch.cat([data.x,
                    is_masked.unsqueeze(1),
                    diff_input.unsqueeze(1)], dim=1)

print(f"final feature matrix: {data.x.shape}")
print(f"features: elev,slope,aspect,doubslope,is_masked,diff_input")
print(f"ram: {psutil.virtual_memory().percent}%")

final feature matrix: torch.Size([4792230, 6])
features: elev,slope,aspect,doubslope,is_masked,diff_input
ram: 21.3%


In [7]:
# define GraphSAGE model:

class CanwellSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super(CanwellSAGE, self).__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)

        # regression head
        self.lin1 = Linear(hidden_channels, hidden_channels//2)
        self.lin2 = Linear(hidden_channels//2, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)

        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)

        x = F.relu(self.conv3(x, edge_index))

        x = F.relu(self.lin1(x))
        x = self.lin2(x)

        return x.squeeze(1)

In [8]:
model = CanwellSAGE(in_channels=6, hidden_channels=64).to(device)
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

CanwellSAGE(
  (conv1): SAGEConv(6, 64, aggr=mean)
  (conv2): SAGEConv(64, 64, aggr=mean)
  (conv3): SAGEConv(64, 64, aggr=mean)
  (lin1): Linear(in_features=64, out_features=32, bias=True)
  (lin2): Linear(in_features=32, out_features=1, bias=True)
)
parameters: 19,457


In [9]:
# normalize target y for training stability
y_mean = y[train_idx].mean().item()
y_std  = y[train_idx].std().item()
y_norm = (y-y_mean)/y_std
data.y = y_norm

In [10]:
# set up neighborloader for mini-batch training
train_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10], # sample 10 neighbors per layer
    batch_size=512,
    input_nodes=train_idx,
    shuffle=True,
)

val_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=val_idx,
    shuffle=False,
)

print(f"train batches: {len(train_loader)}")
print(f"val batches:   {len(val_loader)}")
print(f"y mean: {y_mean:.3f}, std: {y_std:.3f}")
print(f"ram: {psutil.virtual_memory().percent:.1f}")

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


train batches: 3881
val batches:   832
y mean: 6.621, std: 15.582
ram: 30.0


In [11]:
# training and validation functions
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

def train():
    model.train()
    total_loss = 0
    count = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        # only compute loss on target nodes (first batch_size nodes)
        out = out[:batch.batch_size]
        target = batch.y[:batch.batch_size]
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        count += batch.batch_size
    return total_loss/count

def validate():
    model.eval()
    total_loss = 0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            out = out[:batch.batch_size]
            target = batch.y[:batch.batch_size]
            loss = criterion(out, target)
            total_loss += loss.item() * batch.batch_size
            count += batch.batch_size
        return total_loss/count

print("train and val functions ready")

train and val functions ready


In [12]:
# training loop with early stopping
best_val_loss = float('inf')
best_model = None
patience = 10
no_improve = 0
max_epochs = 100
train_losses = []
val_losses = []

for epoch in range(max_epochs):
    train_loss = train()
    val_loss   = validate()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0:
        print(f"epoch: {epoch:03d} | train mse: {train_loss:.4f} | val mse: {val_loss:.4f}")

    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_model = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        print(f" new best: {best_val_loss:.4f} at epoch: {epoch}")
        no_improve = 0
    else:
        no_improve+=1

    if no_improve >= patience:
        print(f"\nearly stopping at epoch {epoch}")
        print(f"best epoch: {best_epoch}, best val mse: {best_val_loss:.4f}")
        model.load_state_dict(best_model)
        break

print("\ntraining complete")

epoch: 000 | train mse: 0.1462 | val mse: 0.1147
 new best: 0.1147 at epoch: 0
 new best: 0.1112 at epoch: 1
 new best: 0.1087 at epoch: 3
 new best: 0.1084 at epoch: 4
epoch: 005 | train mse: 0.1003 | val mse: 0.1055
 new best: 0.1055 at epoch: 5
 new best: 0.1045 at epoch: 6
 new best: 0.1027 at epoch: 7
epoch: 010 | train mse: 0.0954 | val mse: 0.0992
 new best: 0.0992 at epoch: 10
 new best: 0.0981 at epoch: 14
epoch: 015 | train mse: 0.0927 | val mse: 0.0989
 new best: 0.0974 at epoch: 17
 new best: 0.0963 at epoch: 18
 new best: 0.0950 at epoch: 19
epoch: 020 | train mse: 0.0908 | val mse: 0.0964
epoch: 025 | train mse: 0.0897 | val mse: 0.0970
 new best: 0.0945 at epoch: 28
epoch: 030 | train mse: 0.0887 | val mse: 0.0947
 new best: 0.0940 at epoch: 33
epoch: 035 | train mse: 0.0878 | val mse: 0.0934
 new best: 0.0934 at epoch: 35
 new best: 0.0907 at epoch: 36
epoch: 040 | train mse: 0.0871 | val mse: 0.0917
 new best: 0.0900 at epoch: 44
epoch: 045 | train mse: 0.0866 | val ms

In [13]:
test_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=test_idx,
    shuffle=False,
)

In [14]:
model.eval()
preds = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        out = out[:batch.batch_size]
        tgt = batch.y[:batch.batch_size]
        preds.append(out.cpu())
        targets.append(tgt.cpu())

preds   = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()

# denormalize
preds_m   = preds   * y_std + y_mean
targets_m = targets * y_std + y_mean

mae  = np.mean(np.abs(preds_m - targets_m))
rmse = np.sqrt(np.mean((preds_m - targets_m)**2))
r2   = 1 - np.sum((targets_m - preds_m)**2) / np.sum((targets_m - np.mean(targets_m))**2)

print(f"test rmse: {rmse:.3f} m")
print(f"test mae:  {mae:.3f} m")
print(f"test R^2:  {r2:.4f}")

test rmse: 4.648 m
test mae:  3.058 m
test R^2:  0.9108


In [15]:
## map predictions back to raster space
import pickle
import rasterio
from rasterio.transform import from_bounds

# reload spatial info from pickle

with open('../data/canwell_graph.pkl', 'rb') as f:
    graph_data = pickle.load(f)

shape = graph_data['shape']
transform = graph_data['transform']
valid_mask = graph_data['valid_mask']

# build empty reconstruction raster
recon = np.full(shape, np.nan, dtype=np.float32)
true_vals = np.full(shape, np.nan, dtype=np.float32)
mask_raster = np.zeros(shape, dtype=np.uint8)

# reload node_to_idx mapping
node_to_idx = feat['node_to_idx']
idx_to_node = {v: k for k, v in node_to_idx.items()}

# fill test node predictions
for i, node_id in enumerate(test_idx.numpy()):
    r, c = graph_data['G'].nodes[idx_to_node[node_id]]['dem_idx']
    recon[r, c]       = preds_m[i]
    true_vals[r, c]   = targets_m[i]
    mask_raster[r, c] = 1

print(f"filled {np.sum(~np.isnan(recon)):,} prediction pixels")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

filled 425,726 prediction pixels
ram: 99.1%


In [16]:
del graph_data['G']
import gc
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 25.1%


In [17]:
import rasterio
from rasterio.transform import from_origin

meta = {
    'driver': 'GTiff',
    'dtype': 'float32',
    'width': shape[1],
    'height': shape[0],
    'count': 1,
    'crs': graph_data['crs'],
    'transform': transform,
    'nodata': np.nan
}

# save predicted difference DEM
with rasterio.open('canwell_gnn_preds.tif', 'w', **meta) as dst:
    dst.write(recon, 1)
print("saved pred raster")

# save true difference DEM (test nodes only)
with rasterio.open('canwell_gnn_true.tif', 'w', **meta) as dst:
    dst.write(true_vals, 1)
print("saved true raster")

# save mask (1 = test node, 0 = elsewhere)
meta_mask = meta.copy()
meta_mask['dtype'] = 'uint8'
meta_mask['nodata'] = 255
with rasterio.open('canwell_gnn_mask.tif', 'w', **meta_mask) as dst:
    dst.write(mask_raster, 1)
print("saved mask raster")

print(f"ram: {psutil.virtual_memory().percent:.1f}")

saved pred raster
saved true raster
saved mask raster
ram: 25.3


In [18]:
# run inference (model) on all north slope nodes
all_slope_idx = (is_slope & has_diff).nonzero(as_tuple=True)[0]

all_slope_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=all_slope_idx,
    shuffle=False,
)

model.eval()
all_preds=[]

with torch.no_grad():
    for batch in all_slope_loader:
        batch=batch.to(device)
        out = model(batch.x, batch.edge_index)
        out = out[:batch.batch_size]
        all_preds.append(out.cpu())

all_preds = torch.cat(all_preds).numpy()
all_preds_m = all_preds * y_std + y_mean

print(f"preds for {len(all_preds_m):,} slope nodes")
print(f"ram {psutil.virtual_memory().percent:.1f}%")

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


preds for 2,838,168 slope nodes
ram 34.6%


In [23]:
del all_slope_loader, feat
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 30.5%


In [24]:
del data
gc.collect()
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 30.5%


In [27]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 30.5%


In [26]:
# save model
torch.save({
    'model_state_dict': best_model,
    'y_mean': y_mean,
    'y_std': y_std,
}, 'canwell_sage_model.pt')
print("Model saved.")

# save full slope predictions
torch.save({
    'all_slope_idx': all_slope_idx,
    'all_preds_m': all_preds_m,
    'node_to_idx': node_to_idx,
    'idx_to_node': idx_to_node,
}, 'canwell_slope_predictions.pt')
print("Predictions saved.")

# save test results
torch.save({
    'test_idx': test_idx,
    'preds_m': preds_m,
    'targets_m': targets_m,
}, 'canwell_test_results.pt')
print("Test results saved.")

Model saved.
Predictions saved.
Test results saved.


In [29]:
# reload G just to extract dem_idx mapping
with open('../data/canwell_graph.pkl', 'rb') as f:
    graph_data = pickle.load(f)

G_temp = graph_data['G']
shape = graph_data['shape']
transform = graph_data['transform']

# build idx -> (row, col) lookup
print("Building dem_idx lookup...")
idx_to_demidx = {}
for node_id, attr in G_temp.nodes(data=True):
    idx = node_to_idx[node_id]
    idx_to_demidx[idx] = attr['dem_idx']

del G_temp, graph_data
gc.collect()
print(f"Lookup built for {len(idx_to_demidx):,} nodes")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

Building dem_idx lookup...
Lookup built for 4,792,230 nodes
RAM: 43.9%


In [30]:
# rasterize
recon_full = np.full(shape, np.nan, dtype=np.float32)

for i, node_id in enumerate(all_slope_idx.numpy()):
    r, c = idx_to_demidx[node_id]
    recon_full[r, c] = all_preds_m[i]

print(f"filled {np.sum(~np.isnan(recon_full)):,} pixels")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

with rasterio.open('canwell_gnn_allslopepreds.tif', 'w', **meta) as dst:
    dst.write(recon_full, 1)
print("saved allslopepreds")

filled 2,838,168 pixels
ram: 44.2%
saved allslopepreds
